# Unified Evaluation: Accuracy + Performance

Run **multi-benchmark Korean MCQ evaluations** and **GuideLLM performance benchmarks** in unified MLflow experiments.
For single-benchmark evaluation, see **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb**.


This notebook runs LLM evaluations through the [EvalHub](https://github.com/eval-hub/eval-hub) REST API using the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) Python client. All results are automatically tracked in **MLflow**.

## Why use EvalHub?

| Feature | LMEvalJob (direct) | EvalHub (Phase 2) |
|---------|----------------------|-------------------|
| Interface | Kubernetes CR (YAML) | Python SDK / REST API |
| Frameworks | lm-evaluation-harness only | lm-eval, RAGAS, LightEval, GuideLLM, ... |
| Multi-benchmark | One task per CR | Multiple benchmarks per request |
| Experiment tracking | Manual (Pod logs) | **Built-in MLflow** (metrics, params, artifacts) |
| Result management | `oc get lmevaljob` | Centralized API + MLflow UI |
| Job management | `oc delete lmevaljob` | SDK `client.jobs.cancel()` |

## Prerequisites

- **0_setup/0_model_deploy.ipynb** completed (model deployed)
- **0_setup/1_LMEval_setup.ipynb** completed (RBAC and secrets)
- **0_setup/2_eval_hub_setup.ipynb** completed (EvalHub SDK installed and verified)
- EvalHub service running on the cluster
- MLflow tracking server accessible from EvalHub

## Step 1: Configuration

In [1]:
import os, subprocess
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
import sys; sys.path.insert(0, '..')
from utils.port_forward import ensure_evalhub_port_forward
EVALHUB_URL = ensure_evalhub_port_forward(namespace=NAMESPACE)
_r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LIMIT = int(os.getenv("LIMIT", "5"))

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {BASE_URL}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

EvalHub already reachable at localhost:8443
Namespace:       demo
Model Name:      gemma4-12b
Model Endpoint:  https://gemma4-12b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions
EvalHub URL:     https://localhost:8443
MLflow URI:      https://mlflow.redhat-ods-applications.svc:8443
Sample Limit:    10000


## Step 2: Initialize the EvalHub Client

In [2]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

TLS verification disabled - skipping CA bundle detection


TLS verification disabled (insecure mode)


EvalHub client connected: https://localhost:8443
Model target:             gemma4-12b @ https://gemma4-12b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions


## Step 3: Discover Available Korean Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider and filter for Korean tasks.

In [3]:
import yaml, pathlib, httpx

KOREAN_PROVIDER_YAML = pathlib.Path("../adapters/korean-mcq/provider.yaml")
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"
KOREAN_PROVIDER_ID = None

for p in client.providers.list():
    if p.name == KOREAN_PROVIDER_NAME:
        KOREAN_PROVIDER_ID = p.resource.id
        print(f"Korean MCQ provider already registered (id={KOREAN_PROVIDER_ID})")
        break

if not KOREAN_PROVIDER_ID and KOREAN_PROVIDER_YAML.exists():
    raw = KOREAN_PROVIDER_YAML.read_text().replace("${NAMESPACE}", NAMESPACE)
    provider_def = yaml.safe_load(raw)
    resp = httpx.post(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={
            "Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}",
            "Content-Type": "application/json",
            "X-Tenant": NAMESPACE,
        },
        json=provider_def, verify=False, timeout=10,
    )
    if resp.status_code in (200, 201):
        data = resp.json()
        KOREAN_PROVIDER_ID = data["resource"]["id"]
        print(f"Korean MCQ provider registered (id={KOREAN_PROVIDER_ID}, benchmarks={len(data.get('benchmarks', []))})")
    else:
        print(f"WARNING: Korean MCQ registration failed ({resp.status_code}): {resp.text[:200]}")

all_benchmarks = client.benchmarks.list()

korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "hrm8k"]
korean_benchmarks = [
    bm for bm in all_benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Total benchmarks available: {len(all_benchmarks)}")
print(f"Korean benchmarks found:    {len(korean_benchmarks)}")
print("=" * 70)
for bm in korean_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

Korean MCQ provider already registered (id=korean_mcq)


Total benchmarks available: 203
Korean benchmarks found:    5
  click                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  haerae                                    metrics=[overall_accuracy, category_accuracy]
  kmmlu                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kmmlu_hard                                metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kobest_boolq                              metrics=[overall_accuracy]


---

## Evaluation 1: Multi-Benchmark Korean Evaluation

Submit multiple Korean benchmarks in a single request. EvalHub orchestrates them concurrently and tracks all results under one MLflow experiment.

In [4]:
import time

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
    JobStatus.PARTIALLY_FAILED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")

print("Helper functions defined: wait_for_job, display_job_results")

Helper functions defined: wait_for_job, display_job_results


In [5]:
multi_request = JobSubmissionRequest(
    name="korean-multi-benchmark",
    description="Comprehensive Korean LLM evaluation: KMMLU + CLIcK + HAE-RAE",
    tags=["korean", "comprehensive", "multi-benchmark"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="haerae",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-comprehensive-eval",
        tags=[
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="evaluation_type", value="comprehensive"),
        ],
    ),
)

print("Multi-Benchmark Request:")
print(f"  Name:       {multi_request.name}")
print(f"  Model:      {multi_request.model.name}")
print(f"  Benchmarks: {[b.id for b in multi_request.benchmarks]}")
print(f"  Experiment: {multi_request.experiment.name}")
print(f"\nUncomment the next cell to submit.")

Multi-Benchmark Request:
  Name:       korean-multi-benchmark
  Model:      gemma4-12b
  Benchmarks: ['kmmlu', 'click', 'haerae']
  Experiment: korean-comprehensive-eval

Uncomment the next cell to submit.


In [6]:
# SKIPPED: Use Unified Evaluation (Phase A) below instead
print("Skipped - see Phase A below")

Skipped - see Phase A below


---

## Evaluation 2: Sample Size Comparison

Compare evaluation results with different sample limits on the same benchmark. Each evaluation is tracked as a separate MLflow run within the same experiment for easy comparison.

In [7]:
limit_configs = [
    {"name": "kmmlu-limit-100", "limit": 100},
    {"name": "kmmlu-limit-2000", "limit": 2000},
]

limit_jobs = []
for config in limit_configs:
    request = JobSubmissionRequest(
        name=config["name"],
        description=f"KMMLU evaluation with limit={config['limit']}",
        tags=["korean", "kmmlu", "limit-comparison"],
        model=model,
        benchmarks=[
            BenchmarkConfig(
                id="kmmlu",
                provider_id=KOREAN_PROVIDER_ID,
                parameters={
                    "temperature": 0.0,
                    "max_tokens": 16,
                    "limit": config["limit"],
                },
            ),
        ],
        experiment=ExperimentConfig(
            name="kmmlu-limit-comparison",
            tags=[
                ExperimentTag(key="comparison_type", value="sample-size"),
                ExperimentTag(key="limit", value=str(config["limit"])),
            ],
        ),
    )
    limit_jobs.append(request)
    print(f"Prepared: {config['name']} (limit={config['limit']})")

print(f"\n{len(limit_jobs)} jobs ready. Uncomment below to submit.")

Prepared: kmmlu-limit-100 (limit=100)
Prepared: kmmlu-limit-2000 (limit=2000)

2 jobs ready. Uncomment below to submit.


In [8]:
# SKIPPED: Use Unified Evaluation (Phase A) below instead
print("Skipped - see Phase A below")

Skipped - see Phase A below


---

## Evaluation 3: Unified Evaluation (Accuracy + Performance)

Run **Korean MCQ accuracy benchmarks** and **GuideLLM performance benchmarks** together, tracked under a single MLflow experiment. Accuracy jobs run first to avoid load interference with performance measurements.

In [9]:
from datetime import datetime

ts = datetime.now().strftime("%m%d-%H%M")
EXPERIMENT_NAME = f"{MODEL_NAME}-full-eval-{ts}"

# GuideLLM needs the base /v1 endpoint
GUIDELLM_URL = BASE_URL.replace("/v1/completions", "/v1").replace("/v1/chat/completions", "/v1")
guidellm_model = ModelConfig(url=GUIDELLM_URL, name=MODEL_NAME)

print(f"Unified experiment: {EXPERIMENT_NAME}")
print(f"Korean MCQ URL:     {BASE_URL}")
print(f"GuideLLM URL:       {GUIDELLM_URL}")

# --- Phase A: Korean MCQ Accuracy (5 benchmarks) ---
accuracy_benchmarks = ["click", "haerae", "kmmlu", "kmmlu_hard", "kobest_boolq"]
accuracy_jobs = []

for bm in accuracy_benchmarks:
    req = JobSubmissionRequest(
        name=f"unified-{bm}-{ts}",
        description=f"Korean MCQ accuracy: {bm}",
        tags=["unified", "accuracy", MODEL_NAME],
        model=model,
        benchmarks=[
            BenchmarkConfig(
                id=bm,
                provider_id=KOREAN_PROVIDER_ID,
                parameters={"limit": 10000, "tokenizer": os.getenv("TOKENIZER", "")},
            )
        ],
        experiment=ExperimentConfig(name=EXPERIMENT_NAME),
    )
    job = client.jobs.submit(req)
    accuracy_jobs.append((bm, job.id))
    print(f"  [{bm}] submitted: {job.id}")

print(f"\n{len(accuracy_jobs)} accuracy jobs submitted.")

Unified experiment: gemma4-12b-full-eval-0611-0959
Korean MCQ URL:     https://gemma4-12b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions
GuideLLM URL:       https://gemma4-12b-kserve-workload-svc.demo.svc.cluster.local:8000/v1


  [click] submitted: 9b86f1ce-e871-4a9d-b4b4-fe52938f2bbb


  [haerae] submitted: c1a4a62b-4d1c-404d-9381-3e7f03c5e879


  [kmmlu] submitted: 0cbfc12d-b10b-4b5b-be26-2b9988b385e3


  [kmmlu_hard] submitted: 492f0e52-237c-4637-b0bd-e1557c7c94fc


  [kobest_boolq] submitted: a28b796d-4da8-4ab9-ba22-15195612c49e

5 accuracy jobs submitted.


In [10]:
# Wait for all accuracy jobs to complete
import time

print("Waiting for accuracy jobs...")
for i in range(120):
    states = {}
    for bm, jid in accuracy_jobs:
        j = client.jobs.get(jid)
        states[bm] = j.state.value
    
    done = all(s in ("completed", "failed", "error") for s in states.values())
    status_str = " | ".join(f"{k}={v}" for k, v in states.items())
    print(f"  [{i*15}s] {status_str}")
    
    if done:
        print("\nAll accuracy jobs finished!")
        break
    time.sleep(15)

# Display accuracy results
for bm, jid in accuracy_jobs:
    j = client.jobs.get(jid)
    if hasattr(j, "results") and j.results:
        for r in j.results.benchmarks:
            acc = r.metrics.get("overall_accuracy", "N/A")
            print(f"  {bm:15s} | accuracy={acc}% | mlflow_run={r.mlflow_run_id}")

Waiting for accuracy jobs...


  [0s] click=pending | haerae=pending | kmmlu=pending | kmmlu_hard=pending | kobest_boolq=pending


  [15s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


  [30s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


  [45s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


...

  click           | accuracy=73.88% | mlflow_run=1d7c84e078cf4e23a7896f7c1b84aab3


  haerae          | accuracy=69.87% | mlflow_run=e72886dea8d949888e8777f98db8dd83


  kmmlu           | accuracy=57.51% | mlflow_run=0d07861cb4604eb4a6a979bf65914bed


  kmmlu_hard      | accuracy=33.8% | mlflow_run=4533f50fbec949b0ad5b48d4e699c3c2


  kobest_boolq    | accuracy=96.08% | mlflow_run=4d6f7d916e134f309e208535ad52c34c


### Phase B: GuideLLM Performance

After accuracy jobs complete, run a GuideLLM performance test against the same model. Results are tracked in the **same MLflow experiment** for unified visibility.

In [11]:
# --- Phase B: GuideLLM Performance ---
# Use 'throughput' profile to discover max throughput without specifying a fixed rate.
# This avoids "Invalid rates in sweep" errors with large models (e.g. 32B+).
perf_request = JobSubmissionRequest(
    name=f"unified-perf-{MODEL_NAME}-{ts}",
    description=f"GuideLLM throughput test for {MODEL_NAME}",
    tags=["unified", "performance", "guidellm", MODEL_NAME],
    model=guidellm_model,
    benchmarks=[
        BenchmarkConfig(
            id="throughput",
            provider_id="guidellm",
            parameters={
                "max_seconds": 180,
                "max_requests": 50,
                "data": "prompt_tokens=128,output_tokens=64",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=EXPERIMENT_NAME),
)

perf_job = client.jobs.submit(perf_request)
print(f"GuideLLM job submitted: {perf_job.id}")
print(f"Experiment: {EXPERIMENT_NAME}")

# Wait for performance job
for i in range(60):
    j = client.jobs.get(perf_job.id)
    state = j.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nGuideLLM job {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(j, "results") and j.results:
    for bm in j.results.benchmarks:
        print(f"\nPerformance metrics (MLflow run: {bm.mlflow_run_id}):")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

print(f"\n--- Unified evaluation complete ---")
print(f"All results tracked in MLflow experiment: {EXPERIMENT_NAME}")

HTTPStatusError: Client error '404 Not Found' for url 'https://localhost:8443/api/v1/evaluations/jobs'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

### Log GuideLLM Metrics to MLflow (Manual)

The GuideLLM adapter does **not** automatically create MLflow runs — it returns metrics to the EvalHub API but skips the MLflow logging step (unlike the Korean MCQ adapter which uses `mlflow.log_metrics()` internally).

To keep all results visible in a single MLflow experiment, we manually log the GuideLLM metrics here.

In [12]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Find the GuideLLM completed job from EvalHub
guidellm_metrics = None
guidellm_job_name = None
for j_item in client.jobs.list():
    if j_item.effective_state == JobStatus.COMPLETED and j_item.results:
        for bm in j_item.results.benchmarks:
            if bm.id in ("throughput", "constant", "sweep", "quick_perf_test") and bm.mlflow_run_id is None:
                guidellm_metrics = bm.metrics
                guidellm_job_name = j_item.name
                break
    if guidellm_metrics:
        break

if guidellm_metrics:
    try:
        exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        if exp is None:
            exp_id = mlflow.create_experiment(EXPERIMENT_NAME)
        else:
            exp_id = exp.experiment_id

        with mlflow.start_run(experiment_id=exp_id, run_name=f"guidellm-{guidellm_job_name}"):
            mlflow.set_tag("benchmark", "guidellm-throughput")
            mlflow.set_tag("model", MODEL_NAME)
            mlflow.set_tag("source", "evalhub-manual-log")
            mlflow.log_params({
                "profile": "throughput",
                "max_seconds": 180,
                "max_requests": 50,
                "data": "prompt_tokens=128,output_tokens=64",
            })
            mlflow.log_metrics(guidellm_metrics)
            print(f"GuideLLM metrics logged to MLflow experiment: {EXPERIMENT_NAME}")
            print(f"  Run name: guidellm-{guidellm_job_name}")
            print(f"  Metrics: {guidellm_metrics}")
    except Exception as e:
        print(f"MLflow logging skipped (not reachable from local): {e}")
        print(f"GuideLLM metrics (from EvalHub):")
        for k, v in sorted(guidellm_metrics.items()):
            print(f"  {k}: {v}")
else:
    print("No GuideLLM results found to log. Run Phase B first.")

No GuideLLM results found to log. Run Phase B first.


---

## Step 4: Results Comparison

Compare results across multiple evaluation jobs. Collect metrics from completed jobs and display as a comparison table.

In [13]:
def collect_results_table(client, job_ids=None):
    """Collect benchmark metrics from multiple jobs into a comparison dict.

    Returns: {benchmark_id: {job_name: {metric: value}}}
    """
    if job_ids is None:
        jobs_list = client.jobs.list()
        jobs = [
            j for j in jobs_list
            if j.effective_state == JobStatus.COMPLETED
        ]
    else:
        jobs = [client.jobs.get(jid) for jid in job_ids]

    table = {}
    for j in jobs:
        if not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics

    return table


comparison = collect_results_table(client)

if comparison:
    print("Results Comparison")
    print("=" * 70)
    for benchmark_id, job_results in comparison.items():
        print(f"\n  Benchmark: {benchmark_id}")
        print(f"  {'-' * 60}")
        for job_name, metrics in job_results.items():
            print(f"    {job_name}:")
            for metric, value in metrics.items():
                if isinstance(value, float):
                    print(f"      {metric:30s} = {value:.4f}")
                else:
                    print(f"      {metric:30s} = {value}")
else:
    print("No completed jobs found. Submit and complete evaluations first.")

Results Comparison

  Benchmark: kmmlu_hard
  ------------------------------------------------------------
    unified-kmmlu_hard-0611-0946:
      category_accuracy.accounting   = 54.3500
      category_accuracy.agricultural_sciences = 30
      category_accuracy.aviation_engineering_and_maintenance = 31
      category_accuracy.biology      = 27
      category_accuracy.chemical_engineering = 29
      category_accuracy.chemistry    = 47
      category_accuracy.civil_engineering = 30
      category_accuracy.computer_science = 39
      category_accuracy.construction = 27
      category_accuracy.criminal_law = 34
      category_accuracy.ecology      = 27
      category_accuracy.economics    = 47.6200
      category_accuracy.education    = 43.4800
      category_accuracy.electrical_engineering = 28
      category_accuracy.electronics_engineering = 43
      category_accuracy.energy_management = 36
      category_accuracy.environmental_science = 24
      category_accuracy.fashion      = 27
   

### Comparison Table with pandas

In [14]:
try:
    import pandas as pd

    rows = []
    for benchmark_id, job_results in comparison.items():
        for job_name, metrics in job_results.items():
            for metric, value in metrics.items():
                if isinstance(value, (int, float)):
                    rows.append({
                        "benchmark": benchmark_id,
                        "job": job_name,
                        "metric": metric,
                        "value": value,
                    })

    if rows:
        df = pd.DataFrame(rows)
        pivot = df.pivot_table(
            index=["benchmark", "metric"],
            columns="job",
            values="value",
        )
        display(pivot.style.format("{:.4f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No numeric results to display.")

except ImportError:
    print("pandas not available. Install with: pip install pandas")

---

## Step 5: MLflow Results

EvalHub automatically tracks all evaluation results in MLflow. You do **not** need to install or call `mlflow` directly.

### 1. View Experiment List

Open the MLflow UI in your browser (the URL is printed during EvalHub setup) and navigate to the **Experiments** tab.
Each EvalHub job with an `experiment` parameter creates a corresponding MLflow experiment.

![MLflow Experiment List](../images/eval-result-mlflow.png)

### 2. View Detailed Results & Charts

Click on an experiment to see individual runs. Each run contains:

- **Parameters**: model name, benchmark ID, provider, sample limit
- **Metrics**: `overall_accuracy`, `category_accuracy.*`, `supercategory_accuracy.*`
- **Artifacts**: evaluation logs
- **Traces** (if MLflow Tracing is enabled): individual LLM call prompts/responses

Use the MLflow **Chart** view to compare metrics across runs:
1. Select multiple runs in the experiment
2. Click the **Chart** tab
3. Choose metrics (e.g. `overall_accuracy`) to visualize as bar charts or scatter plots

> **Tip**: The MLflow run ID for each benchmark is available via `job.results.benchmarks[].mlflow_run_id` from the EvalHub SDK.

---

## Step 6: Export Results

### Export to Markdown

In [15]:
def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

## EvalHub Benchmark Results

### kmmlu_hard

| Metric | unified-kmmlu_hard-0611-0946 | unified-kmmlu_hard-0611-0959 |
|---|---|---|
| category_accuracy.accounting | 54.3500 | 54.3500 |
| category_accuracy.agricultural_sciences | 30 | 31 |
| category_accuracy.aviation_engineering_and_maintenance | 31 | 31 |
| category_accuracy.biology | 27 | 27 |
| category_accuracy.chemical_engineering | 29 | 28 |
| category_accuracy.chemistry | 47 | 47 |
| category_accuracy.civil_engineering | 30 | 29 |
| category_accuracy.computer_science | 39 | 39 |
| category_accuracy.construction | 27 | 27 |
| category_accuracy.criminal_law | 34 | 34 |
| category_accuracy.ecology | 27 | 27 |
| category_accuracy.economics | 47.6200 | 47.6200 |
| category_accuracy.education | 43.4800 | 43.4800 |
| category_accuracy.electrical_engineering | 28 | 28 |
| category_accuracy.electronics_engineering | 43 | 41 |
| category_accuracy.energy_management | 36 | 36 |
| category_accuracy.environmental_science | 24 | 24 |
| catego

### Save Results to JSON

In [16]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.effective_state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

Saved: ../results/gemma4-12b/unified-kmmlu_hard-0611-0946_d8c26ffe.json
Saved: ../results/gemma4-12b/unified-haerae-0611-0959_c1a4a62b.json
Saved: ../results/gemma4-12b/unified-kobest_boolq-0611-0959_a28b796d.json
Saved: ../results/gemma4-12b/unified-click-0611-0959_9b86f1ce.json
Saved: ../results/gemma4-12b/unified-kobest_boolq-0611-0946_93853a6a.json
Saved: ../results/gemma4-12b/unified-kmmlu-0611-0946_8ee8dfd2.json
Saved: ../results/gemma4-12b/unified-click-0611-0946_4e1b0794.json
Saved: ../results/gemma4-12b/unified-kmmlu_hard-0611-0959_492f0e52.json
Saved: ../results/gemma4-12b/unified-haerae-0611-0946_1e554087.json
Saved: ../results/gemma4-12b/unified-kmmlu-0611-0959_0cbfc12d.json

10 result(s) saved to /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results/<model-name>/


---

## Summary

This notebook demonstrated advanced EvalHub evaluation workflows:

1. **Multi-benchmark** evaluation in a single request
2. **Sample size comparison** tracked under one MLflow experiment
3. **Unified evaluation** -- Korean MCQ accuracy + GuideLLM performance in one experiment
4. **Results comparison** with pandas DataFrames

5. **Export** -- Markdown and JSON output

### Next Steps

- **2_eval_hub_kmcq_benchmark/2_summarize_results.ipynb** -- Aggregate and generate reports
- **1_eval_hub_guidellm_benchmark/1_guidellm_benchmark.ipynb** -- Standalone GuideLLM profiling
